In [0]:
# Gold answers questions such as:
    # 1. What is the average power?
    # 2. What is the average flow?
    # 3. What is the average temperature difference?
    # 4. How many valid records do we have?
    # 5. What is the daily system performance?

# Build 2 Gold tables:

# Table A: Interval KPI
# One row per timestamp snapshot
# Name: hvacapp_dev3.curated.gold_hvac_interval_kpi

# Table B: Daily KPI
# One row per day
# Name: hvacapp_dev3.curated.gold_hvac_daily_kpi

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hvacapp_dev3.curated;

In [0]:
%sql

-- This table is built from silver_telemetry_clean.
-- Each row represents one clean HVAC system snapshot.
-- Example:
-- event_ts	| supply_temp_c	| return_temp_c	| delta_temp_c | flow_lpm |	power_kw
-- 2026-03-27 10:01:00 |	8.5	| 12.0 |	3.5 |	780	| 350
-- This is already dashboard-ready.

CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_interval_kpi (
  event_ts TIMESTAMP,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  supply_temp_c DOUBLE,
  return_temp_c DOUBLE,
  delta_temp_c DOUBLE,
  flow_lpm DOUBLE,
  power_kw DOUBLE,
  pump_speed_pct DOUBLE,
  record_status STRING,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- Insert data from silver_clean into Gold Interval KPI

DELETE FROM hvacapp_dev3.curated.gold_hvac_interval_kpi;

INSERT INTO hvacapp_dev3.curated.gold_hvac_interval_kpi
SELECT
  event_ts,
  site_id,
  building_id,
  equipment_id AS hvac_system_id,
  chw_supply_temp_c AS supply_temp_c,
  chw_return_temp_c AS return_temp_c,
  (chw_return_temp_c - chw_supply_temp_c) AS delta_temp_c,
  chw_flow_lpm AS flow_lpm,
  hvac_power_kw AS power_kw,
  pump_speed_pct,
  'VALID' AS record_status,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.refined.silver_telemetry_clean;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_interval_kpi
ORDER BY event_ts DESC;

In [0]:
%sql
-- Now create daily KPI table, data is from gold_interval

CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_daily_kpi (
  event_date DATE,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  avg_supply_temp_c DOUBLE,
  avg_return_temp_c DOUBLE,
  avg_delta_temp_c DOUBLE,
  avg_flow_lpm DOUBLE,
  avg_power_kw DOUBLE,
  avg_pump_speed_pct DOUBLE,
  record_count BIGINT,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
-- now insert gold interval kpi data into daily kpi table

DELETE FROM hvacapp_dev3.curated.gold_hvac_daily_kpi;

INSERT INTO hvacapp_dev3.curated.gold_hvac_daily_kpi
SELECT
  CAST(event_ts AS DATE) AS event_date,
  site_id,
  building_id,
  hvac_system_id,
  AVG(supply_temp_c) AS avg_supply_temp_c,
  AVG(return_temp_c) AS avg_return_temp_c,
  AVG(delta_temp_c) AS avg_delta_temp_c,
  AVG(flow_lpm) AS avg_flow_lpm,
  AVG(power_kw) AS avg_power_kw,
  AVG(pump_speed_pct) AS avg_pump_speed_pct,
  COUNT(*) AS record_count,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.curated.gold_hvac_interval_kpi
GROUP BY
  CAST(event_ts AS DATE),
  site_id,
  building_id,
  hvac_system_id;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_daily_kpi
ORDER BY event_date DESC;

In [0]:
# gold_hvac_interval_kpi
        # detailed charts over time
        # trend lines
        # time-series analysis

# gold_hvac_daily_kpi
        # daily summary
        # management reporting
        # dashboard cards
        # average daily performance

In [0]:
# -- Actual power = what the optimized HVAC is using now
# -- Baseline power = what the system would have used without optimization
# -- Savings = baseline minus actual

# -- power_saving_kw = baseline_power_kw - actual_power_kw
# -- power_saving_pct = power_saving_kw / baseline_power_kw * 100

# -- baseline_power_kw = actual power + extra inefficiency
# -- Example:
#         -- actual power = 320 kW
#         -- baseline power = 350 kW
#         -- saving = 30 kW

# -- A. Interval savings table
# -- One row per timestamp
# -- hvacapp_dev3.curated.gold_hvac_interval_savings

# -- B. Daily savings table
# -- One row per day
# -- hvacapp_dev3.curated.gold_hvac_daily_savings

In [0]:
%sql
CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_interval_savings (
  event_ts TIMESTAMP,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  supply_temp_c DOUBLE,
  return_temp_c DOUBLE,
  delta_temp_c DOUBLE,
  flow_lpm DOUBLE,
  actual_power_kw DOUBLE,
  baseline_power_kw DOUBLE,
  power_saving_kw DOUBLE,
  power_saving_pct DOUBLE,
  record_status STRING,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DELETE FROM hvacapp_dev3.curated.gold_hvac_interval_savings;

INSERT INTO hvacapp_dev3.curated.gold_hvac_interval_savings
SELECT
  event_ts,
  site_id,
  building_id,
  hvac_system_id,
  supply_temp_c,
  return_temp_c,
  delta_temp_c,
  flow_lpm,
  actual_power_kw,
  baseline_power_kw,
  ROUND(baseline_power_kw - actual_power_kw, 2) AS power_saving_kw,
  ROUND(((baseline_power_kw - actual_power_kw) / baseline_power_kw) * 100, 2) AS power_saving_pct,
  'VALID' AS record_status,
  current_timestamp() AS created_ts
FROM (
  SELECT
    event_ts,
    site_id,
    building_id,
    equipment_id AS hvac_system_id,
    chw_supply_temp_c AS supply_temp_c,
    chw_return_temp_c AS return_temp_c,
    (chw_return_temp_c - chw_supply_temp_c) AS delta_temp_c,
    chw_flow_lpm AS flow_lpm,
    hvac_power_kw AS actual_power_kw,
    ROUND(hvac_power_kw * (1 + (0.08 + rand() * 0.10)), 2) AS baseline_power_kw
  FROM hvacapp_dev3.refined.silver_telemetry_clean
) t;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_interval_savings
ORDER BY event_ts DESC;

In [0]:
%sql
-- Create daily saving table
CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_daily_savings (
  event_date DATE,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  avg_actual_power_kw DOUBLE,
  avg_baseline_power_kw DOUBLE,
  avg_power_saving_kw DOUBLE,
  avg_power_saving_pct DOUBLE,
  total_records BIGINT,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DELETE FROM hvacapp_dev3.curated.gold_hvac_daily_savings;

INSERT INTO hvacapp_dev3.curated.gold_hvac_daily_savings
SELECT
  CAST(event_ts AS DATE) AS event_date,
  site_id,
  building_id,
  hvac_system_id,
  ROUND(AVG(actual_power_kw), 2) AS avg_actual_power_kw,
  ROUND(AVG(baseline_power_kw), 2) AS avg_baseline_power_kw,
  ROUND(AVG(power_saving_kw), 2) AS avg_power_saving_kw,
  ROUND(AVG(power_saving_pct), 2) AS avg_power_saving_pct,
  COUNT(*) AS total_records,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.curated.gold_hvac_interval_savings
GROUP BY
  CAST(event_ts AS DATE),
  site_id,
  building_id,
  hvac_system_id;


In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_daily_savings
ORDER BY event_date DESC;

In [0]:
%sql
-- # Power is measured in kW.
-- # Energy is usually measured in kWh.

-- # If readings are at 1-minute interval, then:
-- # energy_saved_kwh = power_saving_kw / 60

-- # If readings are at 5-minute interval:
-- # energy_saved_kwh = power_saving_kw * 5 / 60

-- # For now, assume snapshots represent 1 minute intervals.

CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_daily_savings_v2 (
  event_date DATE,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  avg_actual_power_kw DOUBLE,
  avg_baseline_power_kw DOUBLE,
  avg_power_saving_kw DOUBLE,
  avg_power_saving_pct DOUBLE,
  estimated_energy_saved_kwh DOUBLE,
  total_records BIGINT,
  created_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DELETE FROM hvacapp_dev3.curated.gold_hvac_daily_savings_v2;

INSERT INTO hvacapp_dev3.curated.gold_hvac_daily_savings_v2
SELECT
  CAST(event_ts AS DATE) AS event_date,
  site_id,
  building_id,
  hvac_system_id,
  ROUND(AVG(actual_power_kw), 2) AS avg_actual_power_kw,
  ROUND(AVG(baseline_power_kw), 2) AS avg_baseline_power_kw,
  ROUND(AVG(power_saving_kw), 2) AS avg_power_saving_kw,
  ROUND(AVG(power_saving_pct), 2) AS avg_power_saving_pct,

  -- assume each row is 1 minute
  ROUND(SUM(power_saving_kw / 60.0), 2) AS estimated_energy_saved_kwh,

  COUNT(*) AS total_records,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.curated.gold_hvac_interval_savings
GROUP BY
  CAST(event_ts AS DATE),
  site_id,
  building_id,
  hvac_system_id;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_daily_savings_v2
ORDER BY event_date DESC;

In [0]:
%sql
-- tariff = RM 0.43 per kWh
-- estimated_cost_saving_rm = estimated_energy_saved_kwh * 0.43

CREATE OR REPLACE TABLE hvacapp_dev3.curated.gold_hvac_daily_savings_v3 (
  event_date DATE,
  site_id STRING,
  building_id STRING,
  hvac_system_id STRING,
  avg_actual_power_kw DOUBLE,
  avg_baseline_power_kw DOUBLE,
  avg_power_saving_kw DOUBLE,
  avg_power_saving_pct DOUBLE,
  estimated_energy_saved_kwh DOUBLE,
  estimated_cost_saved_rm DOUBLE,
  total_records BIGINT,
  created_ts TIMESTAMP
)
USING DELTA;


In [0]:
%sql
DELETE FROM hvacapp_dev3.curated.gold_hvac_daily_savings_v3;

INSERT INTO hvacapp_dev3.curated.gold_hvac_daily_savings_v3
SELECT
  CAST(event_ts AS DATE) AS event_date,
  site_id,
  building_id,
  hvac_system_id,
  ROUND(AVG(actual_power_kw), 2) AS avg_actual_power_kw,
  ROUND(AVG(baseline_power_kw), 2) AS avg_baseline_power_kw,
  ROUND(AVG(power_saving_kw), 2) AS avg_power_saving_kw,
  ROUND(AVG(power_saving_pct), 2) AS avg_power_saving_pct,
  ROUND(SUM(power_saving_kw / 60.0), 2) AS estimated_energy_saved_kwh,
  ROUND(SUM((power_saving_kw / 60.0) * 0.43), 2) AS estimated_cost_saved_rm,
  COUNT(*) AS total_records,
  current_timestamp() AS created_ts
FROM hvacapp_dev3.curated.gold_hvac_interval_savings
GROUP BY
  CAST(event_ts AS DATE),
  site_id,
  building_id,
  hvac_system_id;

In [0]:
%sql
SELECT *
FROM hvacapp_dev3.curated.gold_hvac_daily_savings_v3
ORDER BY event_date DESC;

In [0]:
%sql
-- validation: Interval summary
SELECT
  ROUND(AVG(actual_power_kw), 2) AS avg_actual_power_kw,
  ROUND(AVG(baseline_power_kw), 2) AS avg_baseline_power_kw,
  ROUND(AVG(power_saving_kw), 2) AS avg_power_saving_kw,
  ROUND(AVG(power_saving_pct), 2) AS avg_power_saving_pct
FROM hvacapp_dev3.curated.gold_hvac_interval_savings;


In [0]:
%sql
-- validation: daily summary
SELECT
  event_date,
  avg_actual_power_kw,
  avg_baseline_power_kw,
  avg_power_saving_kw,
  avg_power_saving_pct,
  estimated_energy_saved_kwh,
  estimated_cost_saved_rm
FROM hvacapp_dev3.curated.gold_hvac_daily_savings_v3
ORDER BY event_date DESC;

In [0]:
# this is a simulation:
        # baseline is synthetic
        # savings are synthetic
        # cost savings are estimated

# My goal now is to build the data model and logic flow correctly.
# Later, baseline can be replaced by:
        # rule-based static control benchmark
        # historical average
        # ML model prediction